# 🔵 Notebook 5 — Pipeline de Moderación de Contenido en Cascada
> **Combinar reglas, ML y LLM en producción**

En este notebook vamos a:
1. Implementar un pipeline de moderación de 3 capas: reglas → ML → LLM
2. Cada capa solo procesa lo que la anterior no pudo resolver con alta confianza
3. Este patrón **reduce costos** (las capas baratas filtran primero) y **maximiza precisión**

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

## 2. Configuración

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from dataclasses import dataclass
from typing import Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("✅ Configuración lista")

## 3. Dataset de entrenamiento para el clasificador ML

In [ ]:
# Dataset de comentarios etiquetados (tóxico = 1, ok = 0)
comentarios_train = [
    # Tóxicos
    ("Eres un completo idiota, no sirves para nada", 1),
    ("Ojalá te pase algo malo, te lo mereces", 1),
    ("Imbécil, aprende antes de hablar", 1),
    ("Qué estupidez tan grande, inútil", 1),
    ("Me tienes harto con tus comentarios basura", 1),
    ("Eres un fraude y todos lo saben", 1),
    ("Cállate la boca, nadie quiere escucharte", 1),
    ("Tus ideas son una porquería", 1),
    # No tóxicos
    ("Buen punto, no lo había considerado así", 0),
    ("Estoy en desacuerdo pero respeto tu opinión", 0),
    ("¿Podrías explicar mejor ese argumento?", 0),
    ("Interesante perspectiva, aunque yo lo veo diferente", 0),
    ("Gracias por compartir esta información", 0),
    ("Me parece que hay puntos a revisar en tu análisis", 0),
    ("Muy buen aporte al debate", 0),
    ("Creo que hay formas más constructivas de verlo", 0),
    # Ambiguos (para que el LLM brille)
    ("Este producto es una basura total", 1),
    ("No entiendo cómo alguien puede ser tan torpe", 1),
    ("Qué decepción, esperaba más de ti", 0),
    ("Nunca más compro en esta tienda basura", 1),
]

df_train = pd.DataFrame(comentarios_train, columns=["texto", "toxico"])
print(f"Dataset de entrenamiento: {df_train.shape}")
print(f"Tasa de toxicidad: {df_train['toxico'].mean():.1%}")

## 4. Capa 1 — Filtro por reglas (palabras clave)

In [ ]:
PALABRAS_PROHIBIDAS = [
    "idiota", "imbécil", "inútil", "estúpido", "basura",
    "maldito", "imbecil", "hdp", "cretino", "animal"
]

@dataclass
class ResultadoModeracion:
    texto: str
    decision: str  # "rechazado" | "aprobado" | "pendiente"
    capa: str      # "reglas" | "ml" | "llm"
    confianza: float
    razon: Optional[str] = None

def capa_reglas(texto: str) -> Optional[ResultadoModeracion]:
    """
    Capa 1: Detecta palabras prohibidas explícitas.
    Retorna resultado si puede decidir con alta confianza, None si necesita análisis.
    """
    texto_lower = texto.lower()
    palabras_encontradas = [p for p in PALABRAS_PROHIBIDAS if p in texto_lower]
    
    if palabras_encontradas:
        return ResultadoModeracion(
            texto=texto,
            decision="rechazado",
            capa="reglas",
            confianza=1.0,
            razon=f"Palabras prohibidas: {palabras_encontradas}"
        )
    return None  # No puede decidir → pasa a siguiente capa

# Test
print(capa_reglas("Eres un idiota"))
print(capa_reglas("Estoy en desacuerdo contigo"))

## 5. Capa 2 — Clasificador ML (TF-IDF + Regresión Logística)

In [ ]:
UMBRAL_CONFIANZA_ML = 0.85  # solo decide si confianza > 85%

# Entrenar el modelo
pipeline_ml = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LogisticRegression(random_state=42, max_iter=1000))
])

X_train = df_train["texto"]
y_train = df_train["toxico"]
pipeline_ml.fit(X_train, y_train)

def capa_ml(texto: str) -> Optional[ResultadoModeracion]:
    """
    Capa 2: Clasificador ML. Solo decide si la confianza supera el umbral.
    """
    proba = pipeline_ml.predict_proba([texto])[0]
    pred = int(np.argmax(proba))
    confianza = float(proba[pred])
    
    if confianza >= UMBRAL_CONFIANZA_ML:
        return ResultadoModeracion(
            texto=texto,
            decision="rechazado" if pred == 1 else "aprobado",
            capa="ml",
            confianza=confianza,
            razon=f"Clasificador ML (confianza: {confianza:.0%})"
        )
    return None  # Confianza insuficiente → pasa a LLM

print("✅ Clasificador ML entrenado")
print(f"  Umbral de confianza: {UMBRAL_CONFIANZA_ML:.0%}")

## 6. Capa 3 — LLM (Gemini) para casos ambiguos

In [ ]:
prompt_moderacion = ChatPromptTemplate.from_template("""
Eres un sistema de moderación de contenido preciso y justo.
Analiza este comentario y determina si debe ser rechazado.

Comentario: "{texto}"

Criterios para rechazar:
- Ataques personales directos
- Amenazas o deseos de daño
- Lenguaje discriminatorio
- Acoso o intimidación

NO rechazar por:
- Críticas constructivas
- Opiniones negativas sobre productos/servicios
- Desacuerdo respetuoso

Responde ÚNICAMENTE con este JSON:
{{"decision": "rechazado" o "aprobado", "confianza": número entre 0.0 y 1.0, "razon": "explicación breve"}}
""")

chain_moderacion = prompt_moderacion | llm | StrOutputParser()

def capa_llm(texto: str) -> ResultadoModeracion:
    """
    Capa 3: LLM para análisis contextual profundo de casos ambiguos.
    Siempre decide (es la última capa).
    """
    raw = chain_moderacion.invoke({"texto": texto})
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    
    return ResultadoModeracion(
        texto=texto,
        decision=data["decision"],
        capa="llm",
        confianza=float(data["confianza"]),
        razon=data["razon"]
    )

print("✅ Capa LLM configurada")

## 7. Pipeline completo en cascada

In [ ]:
def moderar_comentario(texto: str) -> ResultadoModeracion:
    """
    Pipeline de moderación en cascada:
    Reglas → ML → LLM (solo si las anteriores no son suficientes)
    """
    # Capa 1: Reglas
    resultado = capa_reglas(texto)
    if resultado:
        return resultado
    
    # Capa 2: ML
    resultado = capa_ml(texto)
    if resultado:
        return resultado
    
    # Capa 3: LLM (fallback final)
    return capa_llm(texto)


# Casos de prueba
comentarios_test = [
    "Eres el peor idiota que he conocido",             # debería ser → reglas
    "Buen trabajo, me gustó mucho el artículo",        # debería ser → ml aprobado
    "Ojalá te pase algo terrible, lo mereces",         # debería ser → ml rechazado
    "No estoy de acuerdo, pero lo respeto",            # debería ser → ml aprobado
    "Qué decepción, esperaba mucho más de un profesional como tú",  # ambiguo → llm
    "Esta empresa es una estafa, nunca más compro aquí",            # ambiguo → llm
    "Tu análisis ignora factores importantes, reconsidera",         # ambiguo → llm
]

print("\n" + "="*70)
print("🛡️ PIPELINE DE MODERACIÓN EN CASCADA")
print("="*70)

resultados = []
for comentario in comentarios_test:
    r = moderar_comentario(comentario)
    resultados.append(r)
    emoji = "🔴" if r.decision == "rechazado" else "✅"
    print(f"\n{emoji} [{r.capa.upper()}] {r.decision.upper()} ({r.confianza:.0%})")
    print(f"   Texto: {comentario[:60]}..." if len(comentario) > 60 else f"   Texto: {comentario}")
    print(f"   Razón: {r.razon}")

## 8. Métricas del pipeline

In [ ]:
from collections import Counter

capas_usadas = Counter([r.capa for r in resultados])
decisiones = Counter([r.decision for r in resultados])

total = len(resultados)

print("\n" + "="*50)
print("📊 MÉTRICAS DEL PIPELINE")
print("="*50)
print(f"\nTotal comentarios procesados: {total}")

print("\n🏗️ Distribución por capa:")
for capa, count in capas_usadas.items():
    bar = "█" * count
    pct = count/total
    cost_note = " (gratis)" if capa == "reglas" else " (barato)" if capa == "ml" else " (costoso)"
    print(f"  {capa:<8} {bar:<10} {count}/{total} ({pct:.0%}){cost_note}")

print("\n🎯 Distribución por decisión:")
for decision, count in decisiones.items():
    print(f"  {decision:<12} {count}/{total} ({count/total:.0%})")

# Estimar costo relativo (LLM = 100x más caro que ML)
llm_calls = capas_usadas.get("llm", 0)
ml_calls = capas_usadas.get("ml", 0)
print(f"\n💰 Llamadas al LLM (caro): {llm_calls}/{total} = solo el {llm_calls/total:.0%} del total")
print(f"   → Ahorro estimado vs usar solo LLM: ~{(total-llm_calls)/total:.0%} del costo")

## 🎯 Conclusiones

El patrón de **pipeline en cascada** es un estándar de producción:

| Capa | Velocidad | Costo | Cuando usar |
|------|-----------|-------|-------------|
| Reglas | ⚡ Instantáneo | 💚 Gratis | Casos obvios con keywords |
| ML | 🚀 Muy rápido | 💛 Mínimo | Casos con patrones aprendidos |
| LLM | 🐢 Segundos | 🔴 Caro | Casos ambiguos que necesitan razonamiento |

**Patrón clave:** `Reglas → ML → LLM (solo casos ambiguos)` reduce costos hasta un 80-90% vs usar solo LLM